# 04 — Vision-Language Models: Flamingo & LLaVA

## What
Vision-Language Models (VLMs) connect a frozen vision encoder (ViT) to a language model, enabling the LLM to reason about image content. Two dominant architectures: (1) *Flamingo-style*: interleaved cross-attention gates inserted between transformer blocks let the LLM attend to image features at every layer; (2) *LLaVA-style*: a lightweight projection layer maps patch embeddings into the LLM's text token space — simpler and cheaper.

## Why
The key insight is that large LLMs already know how to reason; they just need visual grounding. By freezing both the vision encoder and LLM and only training the connector, VLMs like LLaVA achieve strong visual instruction following with modest compute.

## Community context
Flamingo (DeepMind, 2022) first demonstrated few-shot VQA via cross-attention gating. LLaVA (Haotian Liu et al., 2023) showed that a simple linear projection suffices for many tasks. BLIP-2 added the Q-Former, InstructBLIP added instruction tuning. LLaVA-1.5 replaced the linear projection with a 2-layer MLP and matched much larger models.

In [ ]:
# LLaVA-style architecture: vision encoder + projection + LLM token stream
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

class VisualProjector:
    """
    LLaVA projection: maps ViT patch embeddings to LLM token space.
    LLaVA-1.5 uses 2-layer MLP; original LLaVA uses single linear.
    """
    def __init__(self, vision_dim=256, llm_dim=512):
        self.W1 = np.random.randn(vision_dim, llm_dim) * np.sqrt(2/vision_dim)
        self.W2 = np.random.randn(llm_dim, llm_dim) * np.sqrt(2/llm_dim)
        self.b1 = np.zeros(llm_dim)
        self.b2 = np.zeros(llm_dim)

    def __call__(self, patch_embeds):
        """patch_embeds: (N_patches, vision_dim) -> (N_patches, llm_dim)"""
        h = np.maximum(0, patch_embeds @ self.W1 + self.b1)  # ReLU
        return h @ self.W2 + self.b2

class TokenMixer:
    """
    Concatenates system prompt + visual tokens + user text tokens.
    In real models these feed into the LLM causal transformer.
    """
    def build_sequence(self, sys_tokens, visual_tokens, user_tokens):
        seq = np.vstack([sys_tokens, visual_tokens, user_tokens])
        return seq

    def build_attention_mask(self, n_sys, n_visual, n_user):
        N = n_sys + n_visual + n_user
        # Causal mask: each token can attend to itself and all previous
        mask = np.tril(np.ones((N, N)))
        return mask

np.random.seed(42)
vision_dim = 256
llm_dim    = 512
n_patches  = 16    # 4x4 grid for demo

projector = VisualProjector(vision_dim, llm_dim)
mixer     = TokenMixer()

# Simulated inputs
patch_embeds = np.random.randn(n_patches, vision_dim)     # from ViT
visual_tokens = projector(patch_embeds)                   # projected to LLM space
sys_tokens   = np.random.randn(4,  llm_dim)  # 'You are a helpful assistant...'
user_tokens  = np.random.randn(8,  llm_dim)  # 'What is in this image?'

seq   = mixer.build_sequence(sys_tokens, visual_tokens, user_tokens)
mask  = mixer.build_attention_mask(4, n_patches, 8)

print('VLM Token Sequence Construction (LLaVA-style):')
print(f'  System tokens:      {sys_tokens.shape}')
print(f'  Visual tokens:      {visual_tokens.shape}  (projected from {patch_embeds.shape})')
print(f'  User text tokens:   {user_tokens.shape}')
print(f'  Full sequence:      {seq.shape}')
print(f'  Attention mask:     {mask.shape}  (causal)')
print(f'\nVisual token stats: mean={visual_tokens.mean():.4f} std={visual_tokens.std():.4f}')
print('Visual tokens are interleaved with text in the LLM context window')

## Flamingo Cross-Attention Gates

Flamingo inserts *gated cross-attention dense* layers between every transformer block. Each gate has a learnable tanh-gating scalar `alpha`, initialised to 0 so that at the start of training the model behaves like the original LLM and visual influence is gradually learned.

In [ ]:
# Flamingo-style gated cross-attention
class GatedCrossAttention:
    """
    Language tokens attend to visual tokens via cross-attention.
    tanh(alpha) gates scale controls how much visual influence enters.
    alpha initialised to 0 -> gate starts closed.
    """
    def __init__(self, dim, n_heads=4):
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        scale = np.sqrt(dim)
        self.Wq = np.random.randn(dim, dim) / scale  # lang -> query
        self.Wk = np.random.randn(dim, dim) / scale  # visual -> key
        self.Wv = np.random.randn(dim, dim) / scale  # visual -> value
        self.Wo = np.random.randn(dim, dim) / scale
        self.alpha = 0.0  # tanh gate, learned; init=0 -> gate closed

    def __call__(self, lang_tokens, visual_tokens):
        """lang_tokens: (N_lang, D), visual_tokens: (N_vis, D)"""
        Q = lang_tokens   @ self.Wq   # (N_lang, D)
        K = visual_tokens @ self.Wk   # (N_vis, D)
        V = visual_tokens @ self.Wv
        # Attention from language to visual
        attn  = softmax(Q @ K.T / np.sqrt(self.head_dim))
        xattn = attn @ V @ self.Wo    # (N_lang, D)
        # tanh gate controls visual influence
        gate  = np.tanh(self.alpha)
        output = lang_tokens + gate * xattn
        return output, gate

gca = GatedCrossAttention(dim=512)

lang_toks = np.random.randn(8, 512)
vis_toks  = visual_tokens  # from projector above

print('Gate alpha=0 (untrained): gate =', np.tanh(0.0))
out, gate = gca(lang_toks, vis_toks)
print(f'Visual influence on language tokens: {gate:.4f} (zero initially)')

# After some training, alpha might be ~1.0
gca.alpha = 1.0
out, gate = gca(lang_toks, vis_toks)
print(f'\nAfter training (alpha=1.0): gate = {gate:.4f}')
print(f'Output shape: {out.shape}  (language tokens enriched with visual context)')